# Span Results — All Languages

Runs all analyses across all languages directly from Google Sheets.

**To use:**
1. Run the **Setup** cell — installs planars and asks for Google permission (once per session)
2. Run remaining cells in order, or use **Runtime → Run all**

In [ ]:
# Setup — run once per session
import subprocess, sys, importlib

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'git+https://github.com/jcgood/planars.git',
     'gspread', 'google-auth'])
importlib.invalidate_caches()

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
from googleapiclient.discovery import build

creds, _ = default(scopes=[
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets',
])
gc = gspread.authorize(creds)
drive_svc = build('drive', 'v3', credentials=creds)

print('Ready ✓')

In [ ]:
# Load planars config and all language manifests from Drive.
# CONFIG_FILE_ID is baked in at notebook generation time.
# To update it, re-run: python -m coding generate-notebooks --apply
import json

CONFIG_FILE_ID = '{{CONFIG_FILE_ID}}'
config = json.loads(drive_svc.files().get_media(fileId=CONFIG_FILE_ID).execute())
manifest = config

loaded = [lid for lid in manifest if not lid.startswith('_')]
print(f'Loaded {len(loaded)} language(s): {loaded}')

In [ ]:
# Helper — run once after loading the manifest
from planars.io import load_filled_sheet

def show_class_reports(gc, manifest, class_name, derive_fn, format_fn):
    """Print format_result() output for every construction of class_name across all languages."""
    found_any = False
    for lang_id, lang_data in manifest.items():
        sheet_info = lang_data.get('sheets', {}).get(class_name)
        if not sheet_info:
            continue
        try:
            ss = gc.open_by_key(sheet_info['spreadsheet_id'])
        except Exception as e:
            print(f'  WARNING: could not open {class_name} sheet for {lang_id}: {e}')
            continue
        construction_params = sheet_info.get('construction_params', {})
        for construction in sheet_info['constructions']:
            found_any = True
            sep = '\u2500' * 60
            print(f'\n{sep}')
            print(f'  {class_name}  [{lang_id}]  \u2014  {construction}')
            print(sep)
            try:
                ws = ss.worksheet(construction)
            except Exception:
                print(f'  WARNING: tab not found')
                continue
            required = set(construction_params.get(construction, {}).get('param_names', []))
            loaded = load_filled_sheet(ws, required_params=required, strict=False)
            result = derive_fn(_data=loaded, strict=False)
            print(format_fn(result))
    if not found_any:
        print(f'No {class_name} sheets found in manifest.')

print('Helper defined.')

In [ ]:
# __CLASS_CELLS_MARKER__

---
## Domain Chart

All spans plotted as horizontal segments. Hover over a segment to see position names and span size.

In [ ]:
from planars.charts import collect_all_spans_from_sheets, charts_by_language

df, lang_meta = collect_all_spans_from_sheets(gc, manifest)
for lang_id, fig in charts_by_language(df, lang_meta).items():
    print(f'--- {manifest.get(lang_id, {}).get("glottolog", {}).get("name", lang_id)} [{lang_id}] ---')
    fig.show()